# UK Job Market — Empirical Classification Replay

## Objective

Evaluate the behaviour of an agent-assisted job advertisement
processing pipeline using 60 saved Adzuna advertisements and
their previously captured LLM classifications.

The notebook executes the existing production processing
workflow without requiring live API access.

The analysis examines classification agreement, review
requirements, analytical eligibility and reproducibility.

Classification agreement is not treated as independently
verified classification accuracy.

## Contents

- [Local inputs and capture validation](#local-inputs)
- [Production pipeline replay](#production-replay)
- [Classification disagreements](#disagreements)
- [Analytical eligibility and reconciliation](#eligibility)
- [Deterministic report publication](#publication)
- [Conclusions and limitations](#conclusions)

<a id="local-inputs"></a>

## Local inputs and capture validation

Run from the repository root or `notebooks/` with the installed project, pandas
and Jupyter kernel. The three original snapshots **and metadata sidecars** plus
`data/raw/classification_lesson44.jsonl` are required. The loader checks snapshot
names, source, collection timestamps, population size and excerpt completeness.
Missing or incompatible local inputs fail rather than trigger fresh inference.
This notebook writes four local reports under `data/processed/lesson44/`.
Raw inputs, captured responses and reports remain Git-ignored pending redistribution
review. Clear outputs before saving for Git.


In [ ]:
from pathlib import Path
from collections import Counter
import runpy

import pandas as pd

from uk_job_market_agents.agents.classification_replay import (
    ClassificationReplay,
    read_classification_captures,
)
from uk_job_market_agents.agents.batch_orchestration import (
    process_jobs,
)
from uk_job_market_agents.analytics.job_export import (
    build_job_export_rows,
)
from uk_job_market_agents.analytics.analytical_report import (
    build_analytical_report,
)

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").exists()

ROOT

In [ ]:
capture_runner = runpy.run_path(
    str(ROOT / "scripts" / "capture_classifications.py")
)

postings = capture_runner["load_population"]()

pd.Series(
    {
        "Total postings": len(postings),
        "Unique source identities": len({
            (p.source, p.source_id)
            for p in postings
        }),
    },
    name="Count",
)

In [ ]:
capture_path = (
    ROOT / "data" / "raw"
    / "classification_lesson44.jsonl"
)

captures = read_classification_captures(capture_path)

replay = ClassificationReplay(captures)

assert len(captures) == len(postings) == 60

assert {
    (c.source, c.source_id) for c in captures
} == {
    (p.source, p.source_id) for p in postings
}

for posting in postings:
    replay.classify(posting)

print(f"Validated {len(captures)} compatible captures.")

In [ ]:
classification_df = pd.DataFrame([
    {
        "source_id": capture.source_id,
        "role_family": capture.classification.role_family.value,
        "work_pattern": capture.classification.work_pattern.value,
    }
    for capture in captures
])

pd.crosstab(
    classification_df["role_family"],
    classification_df["work_pattern"],
    margins=True,
)

In [ ]:
postings_by_identity = {
    (p.source, p.source_id): p
    for p in postings
}

hybrid_df = pd.DataFrame([
    {
        "source_id": capture.source_id,
        "title": postings_by_identity[
            (capture.source, capture.source_id)
        ].raw_title,
        "role_family": capture.classification.role_family.value,
        "reasoning": capture.classification.reasoning,
        "description": postings_by_identity[
            (capture.source, capture.source_id)
        ].raw_description,
    }
    for capture in captures
    if capture.classification.work_pattern.value == "hybrid"
])

with pd.option_context("display.max_colwidth", None):
    display(
        hybrid_df[
            ["source_id", "title", "reasoning"]
        ]
    )

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(
        hybrid_df[
            ["source_id", "description", "reasoning"]
        ]
    )

<a id="production-replay"></a>

## Production Pipeline Replay

Replay the captured LLM classifications through the existing
production processing workflow.

Compare deterministic rules with captured interpretations,
identify review requirements and evaluate analytical eligibility.

All source descriptions are excerpts, so production salary
extraction is excluded under the existing completeness policy.

In [ ]:
outcomes = process_jobs(
    postings,
    classifier=replay.classify,
)

assert len(outcomes) == len(postings) == 60

status_counts = pd.Series(
    Counter(
        outcome.status.value
        for outcome in outcomes
    ),
    name="Advertisements",
)

pd.DataFrame({
    "Count": status_counts,
    "Percentage": (
        status_counts / len(postings) * 100
    ).round(2),
})

In [ ]:
suspect_outcome = next(
    outcome
    for outcome in outcomes
    if outcome.job_id == "5967230748"
)

print(
    suspect_outcome.model_dump_json(indent=2)
)

<a id="disagreements"></a>

## Classification Disagreements

Examine the distribution and nature of classification
disagreements between deterministic rules and captured
LLM interpretations.

Review requirements indicate disagreement rather than
established classification error.

In [ ]:
review_df = pd.DataFrame([
    {
        "source_id": outcome.job_id,
        "title": posting.raw_title,
        "review_reason": (
            outcome.result.classification.review_reason.value
        ),
        "llm_role_family": (
            replay.classify(posting).role_family.value
        ),
        "llm_work_pattern": (
            replay.classify(posting).work_pattern.value
        ),
    }
    for posting, outcome in zip(
        postings, outcomes, strict=True
    )
    if outcome.status.value == "review_required"
])

display(
    review_df["review_reason"]
    .value_counts()
    .to_frame("Count")
)

display(review_df)

In [ ]:
from uk_job_market_agents.classification.rules import (
    classify_role_family,
    classify_work_pattern,
)

comparison_df = pd.DataFrame([
    {
        "source_id": posting.source_id,
        "title": posting.raw_title,
        "rule_role": classify_role_family(
            posting.raw_title,
            posting.raw_description,
        ).value,
        "llm_role": replay.classify(
            posting
        ).role_family.value,
        "rule_work_pattern": classify_work_pattern(
            f"{posting.raw_title} {posting.raw_description}"
        ).value,
        "llm_work_pattern": replay.classify(
            posting
        ).work_pattern.value,
        "description": posting.raw_description,
        "reasoning": replay.classify(posting).reasoning,
    }
    for posting in postings
])

disagreements_df = comparison_df.loc[
    comparison_df["source_id"].isin(
        review_df["source_id"]
    )
]

display(
    disagreements_df.drop(
        columns=["description", "reasoning"]
    )
)

In [ ]:
pd.crosstab(
    comparison_df["rule_work_pattern"],
    comparison_df["llm_work_pattern"],
    margins=True,
)

In [ ]:
work_agreement = (
    comparison_df["rule_work_pattern"]
    == comparison_df["llm_work_pattern"]
)

role_agreement = (
    comparison_df["rule_role"]
    == comparison_df["llm_role"]
)

mutually_unknown = (
    (comparison_df["rule_work_pattern"] == "unknown")
    & (comparison_df["llm_work_pattern"] == "unknown")
)

informative_population = ~mutually_unknown

agreement_summary = pd.DataFrame([
    {
        "metric": "Role-family agreement",
        "numerator": role_agreement.sum(),
        "denominator": len(comparison_df),
    },
    {
        "metric": "Work-pattern agreement",
        "numerator": work_agreement.sum(),
        "denominator": len(comparison_df),
    },
    {
        "metric": "Work-pattern agreement excluding mutual unknowns",
        "numerator": (
            work_agreement & informative_population
        ).sum(),
        "denominator": informative_population.sum(),
    },
])

agreement_summary["rate_pct"] = (
    agreement_summary["numerator"]
    / agreement_summary["denominator"]
    * 100
).round(2)

agreement_summary

# Reconcile the diagnostic to production review decisions and the recorded run.
assert list(agreement_summary["numerator"]) == [59, 51, 16]
assert list(agreement_summary["denominator"]) == [60, 60, 25]
assert set(comparison_df.loc[~(role_agreement & work_agreement), "source_id"]) == set(review_df["source_id"])
assert review_df["review_reason"].value_counts().to_dict() == {"work_pattern": 9, "role_family": 1}

display(agreement_summary)


<a id="eligibility"></a>

## Analytical Eligibility and Reconciliation

Evaluate which observations qualify for downstream analysis
under the existing analytical eligibility policies.

Distinguish successful processing, approved role classifications,
known working arrangements and verified annual salaries.

Reconcile the analytical populations against the original
60 source observations.

In [ ]:
rows = build_job_export_rows(
    postings,
    outcomes,
)

report = build_analytical_report(rows)

assert len(rows) == len(postings) == len(outcomes)

export_df = pd.DataFrame([
    row.model_dump(mode="json")
    for row in rows
])

export_df.head()

assert [row.source_id for row in rows] == [p.source_id for p in postings]
assert all(outcome.result is not None and outcome.result.salary_assessment is None for outcome in outcomes)
for posting, row in zip(postings, rows, strict=True):
    assert (row.source, row.collected_at, row.description_completeness) == (
        posting.source, posting.collected_at, posting.description_completeness,
    )
    assert (row.source_salary_min, row.source_salary_max, row.source_salary_predicted) == (
        posting.source_salary_min, posting.source_salary_max, posting.source_salary_predicted,
    )
    assert row.salary_min is None and row.salary_max is None

from uk_job_market_agents.analytics.data_quality import summarize_data_quality
quality = summarize_data_quality(rows)
assert quality.approved_role_coverage.populated == 50
assert quality.known_work_pattern_coverage.populated == 16
assert quality.verified_salary_coverage.populated == 0
display(pd.DataFrame([
    {"metric": name, **getattr(quality, name).model_dump()}
    for name in ("approved_role_coverage", "known_work_pattern_coverage", "verified_salary_coverage")
]))


In [ ]:
print(report.samples.model_dump_json(indent=2))

display(
    pd.DataFrame([
        group.model_dump(mode="json")
        for group in report.groups
    ])
)

In [ ]:
from uk_job_market_agents.analytics.analytical_dataset import select_role_eligible_rows

approved_df = pd.DataFrame([
    row.model_dump(mode="json") for row in select_role_eligible_rows(rows)
])
approved_matrix = pd.crosstab(approved_df["role_family"], approved_df["work_pattern"])
assert approved_matrix.to_dict() == {
    "hybrid": {"analytics_engineering": 3, "data_engineering": 4, "data_science": 9},
    "unknown": {"analytics_engineering": 16, "data_engineering": 10, "data_science": 8},
}
display(pd.crosstab(approved_df["role_family"], approved_df["work_pattern"], margins=True))

<a id="publication"></a>

## Deterministic Report Publication

Publish record-level, batch-level and grouped analytical
outputs using the existing production CSV exporters.

Verify that repeated offline execution produces identical
processing outcomes and byte-identical analytical exports.

In [ ]:
from functools import partial

from uk_job_market_agents.ingestion.offline_workflow import (
    run_offline_workflow,
)
from uk_job_market_agents.analytics.analytical_report_csv import (
    write_analytical_report_csv,
)

output_dir = ROOT / "data" / "processed" / "lesson44"
output_dir.mkdir(parents=True, exist_ok=True)

output_paths = {
    "jobs": output_dir / "jobs.csv",
    "batch_summary": output_dir / "batch_summary.csv",
    "analytical_samples": output_dir / "analytical_samples.csv",
    "role_family_summary": output_dir / "role_family_summary.csv",
}


def publish_reports():
    replayed_outcomes, batch_summary = run_offline_workflow(
        postings,
        processor=partial(
            process_jobs,
            classifier=replay.classify,
        ),
        job_export_path=output_paths["jobs"],
        summary_export_path=output_paths["batch_summary"],
    )

    assert replayed_outcomes == outcomes

    replayed_rows = build_job_export_rows(
        postings,
        replayed_outcomes,
    )

    replayed_report = build_analytical_report(
        replayed_rows
    )

    assert replayed_report == report

    write_analytical_report_csv(
        replayed_report,
        samples_path=output_paths["analytical_samples"],
        groups_path=output_paths["role_family_summary"],
    )

    return batch_summary


batch_summary = publish_reports()

print(batch_summary.model_dump_json(indent=2))

In [ ]:
from hashlib import sha256


def report_fingerprints():
    return {
        name: sha256(path.read_bytes()).hexdigest()
        for name, path in output_paths.items()
    }


first_run = report_fingerprints()

publish_reports()

second_run = report_fingerprints()

assert first_run == second_run

pd.DataFrame([
    {
        "report": name,
        "size_bytes": output_paths[name].stat().st_size,
        "sha256": fingerprint,
        "identical": fingerprint == second_run[name],
    }
    for name, fingerprint in first_run.items()
])

In [ ]:
published_jobs = pd.read_csv(
    output_paths["jobs"],
    dtype={"source_id": str},
)

published_samples = pd.read_csv(
    output_paths["analytical_samples"]
)

published_groups = pd.read_csv(
    output_paths["role_family_summary"]
)

assert len(published_jobs) == 60
assert published_jobs["source_id"].nunique() == 60

assert (
    published_jobs["processing_status"] == "processed"
).sum() == 50

assert (
    published_jobs["processing_status"] == "review_required"
).sum() == 10

assert published_samples.loc[0, "total"] == 60
assert published_samples.loc[0, "role_eligible"] == 50
assert published_samples.loc[0, "salary_eligible"] == 0

assert published_groups["role_count"].sum() == 50
assert published_groups["verified_salary_count"].sum() == 0

display(published_samples)
display(published_groups)

print("Published analytical populations reconciled.")

published_batch = pd.read_csv(output_paths["batch_summary"])
assert published_batch.loc[0, ["total", "processed", "review_required", "failed"]].tolist() == [60, 50, 10, 0]
assert published_jobs["source_id"].tolist() == [p.source_id for p in postings]
assert published_jobs[["salary_min", "salary_max"]].isna().all().all()
assert published_jobs["source_salary_min"].tolist() == [r.source_salary_min for r in rows]
assert published_jobs["source_salary_max"].tolist() == [r.source_salary_max for r in rows]
assert published_jobs["source_salary_predicted"].astype(str).str.lower().tolist() == [str(r.source_salary_predicted).lower() for r in rows]
assert abs(published_samples.loc[0, "role_eligible_rate"] - 50 / 60) < 1e-9
assert published_samples.loc[0, "salary_eligible_rate"] == 0
assert dict(zip(published_groups["role_family"], published_groups["role_count"])) == {
    "analytics_engineering": 19, "data_engineering": 14, "data_science": 17,
}


<a id="conclusions"></a>

## Conclusions

### Classification agreement

The empirical evaluation used 60 advertisements from three previously collected Adzuna snapshots, with one persisted LLM classification per advertisement.

- Role-family agreement: 59/60 (98.33%).
- Work-pattern agreement: 51/60 (85.00%).
- Work-pattern agreement excluding mutually unknown classifications: 16/25 (64.00%).
- Nine advertisements required review for working arrangements and one for role family.

The observed disagreements demonstrate complementary limitations in deterministic keyword-based classification and contextual LLM interpretation. Neither classifier constitutes independently validated ground truth.

### Analytical eligibility

Of the 60 source observations:

- 50 passed classification reconciliation.
- 10 required classification review.
- None encountered processing failures.
- 16 contained approved, known working arrangements.
- None qualified for verified annual salary analysis.

The absence of verified annual salaries reflects the description-completeness policy rather than an absence of salary disclosures.

### Reproducibility

All 60 captured classifications were replayed through the production processing pipeline without requiring further classification API calls.

Repeated execution produced identical processing outcomes and matching SHA-256 fingerprints for all four CSV reports. Published analytical populations reconciled against the original source observations.

### Limitations

- The sample is small, non-random and not representative of the wider UK labour market.
- Distinct advertisement identifiers do not necessarily represent distinct vacancies.
- Classification agreement is not a measure of classification accuracy.
- All source descriptions are excerpts, restricting salary eligibility and potentially limiting classification context.
- Replay depends on compatible local snapshots, classification captures, prompt fingerprints and model identifiers.
- Reproducibility of captured responses does not establish reproducibility of fresh LLM inference.
- The recorded model identifier need not identify an immutable model version.
- SHA-256 fingerprints establish compatibility, not authenticity.
- Capture persistence assumes a single writer. An interrupted physical write may leave malformed JSON requiring intervention; exactly-once API billing is not guaranteed.
- Publication of the four CSV reports is not a single atomic transaction.
- Public redistribution rights for the underlying source observations remain to be established.

### Overall conclusion

The empirical replay demonstrates that captured LLM interpretations can be integrated into a deterministic analytical workflow with explicit review boundaries, traceable eligibility decisions and reproducible reporting.

The principal achievement is not the elimination of uncertainty, but the prevention of unresolved uncertainty from silently entering downstream analytical populations.